In [1]:
import ast
import os
import pandas as pd

def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import time
from tqdm import tqdm
from Bio import SeqIO
import multiprocessing

def find_seq_record(acc_n, max_key, que):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if max_key == seq_record.id:
            continue
        else:
            temp_id = f'{acc_n}-{seq_record.id}'
            seq_record.id = temp_id
            seq_record.description = ''
            que.put(seq_record)
    handle.close()

for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)

    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 32
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)
    
    NMS_count = 0
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        max_key, max_num = max_dict(chr_data | pla_data)
        NMS_count += len(chr_data | pla_data) - 1
        pool.apply_async(find_seq_record, (acc_n, max_key, que))

    pool.close()
    folder = f'/active-data/analysis_results/chr_pla/genus/NMS_replicon/{genus_name}/'
    os.makedirs(folder, exist_ok=True)
    os.chdir(folder)
    temp_ncl_file = open(f'NMS_replicon_nucleotide_seq.fasta', 'w+')
    
    count = 0
    with tqdm(total = NMS_count, desc=f'{genus_name}({NMS_count})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            time.sleep(0.001)
            if not que.empty():
                seq_record = que.get(True)
                SeqIO.write(seq_record, temp_ncl_file, "fasta")
                count += 1
                pbar.update(1)
                if count == NMS_count:
                    break
            else:
                continue
    
    pool.join()

Escherichia(11232): 100%|███████████████████████████████████████| 11.2k/11.2k [03:30<00:00, 53.3B/s]
Klebsiella(11421): 100%|████████████████████████████████████████| 11.4k/11.4k [04:06<00:00, 46.3B/s]
Staphylococcus(2655): 100%|█████████████████████████████████████| 2.65k/2.65k [00:36<00:00, 73.2B/s]
Pseudomonas(798): 100%|█████████████████████████████████████████████| 798/798 [01:25<00:00, 9.38B/s]
Bacillus(2016): 100%|███████████████████████████████████████████| 2.02k/2.02k [01:05<00:00, 30.7B/s]
Salmonella(2471): 100%|█████████████████████████████████████████| 2.47k/2.47k [01:03<00:00, 38.9B/s]
Streptococcus(178): 100%|███████████████████████████████████████████| 178/178 [00:18<00:00, 9.67B/s]
Streptomyces(1102): 100%|███████████████████████████████████████| 1.10k/1.10k [01:09<00:00, 15.8B/s]
Acinetobacter(2509): 100%|██████████████████████████████████████| 2.51k/2.51k [00:33<00:00, 73.9B/s]
Enterococcus(2334): 100%|███████████████████████████████████████| 2.33k/2.33k [00:35<00:00,